In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error,r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.utils import shuffle
features = pd.read_excel('高温合金数据集.xlsx')
label1 = np.array(features['UTS'])
label2 = np.array(features['TYS'])
label3 = np.array(features['Elong'])
features= features.drop('UTS', axis = 1)
features= features.drop('TYS', axis = 1)
features= features.drop('Elong', axis = 1)
features= features.drop('合金的牌号', axis = 1)
n_samples, n_features = features.shape
features = pd.get_dummies(features)
feature_list = list(features.columns)#获取列名
features = np.array(features)#获取影响因子
from sklearn.preprocessing import MinMaxScaler, StandardScaler
features = MinMaxScaler().fit_transform(features)#将进行归一化处理
from sklearn.model_selection import train_test_split
test_ratio = 0.2#按照五比一划分数据集合；4/5的数据用于训练集合，1/5的数据用于验证集合。
SEED = 26 ### the test/train data is checked on this seed, it has similiar distribution to the whole dataset
train_features1, test_features1, train_labels1, test_labels1 = train_test_split(features, label1,
                                                                            test_size = test_ratio,
                                                                            random_state = SEED)
train_features2, test_features2, train_labels2, test_labels2 = train_test_split(features, label2,
                                                                            test_size = test_ratio,
                                                                            random_state = SEED)
train_features3, test_features3, train_labels3, test_labels3 = train_test_split(features, label3,
                                                                            test_size = test_ratio,
                                                                            random_state = SEED)
model_seed = 100 
from sklearn.ensemble import RandomForestRegressor
Model1 = RandomForestRegressor(random_state=model_seed)
Model2 = RandomForestRegressor(random_state=model_seed)
Model3 = RandomForestRegressor(random_state=model_seed)
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
def evaluate(true_labels, pred_labels):
    errors = abs(pred_labels - true_labels)
    MAE = mean_absolute_error(true_labels, pred_labels)
    mape = 100 * np.mean(errors / true_labels)
    r2 = r2_score(true_labels, pred_labels)
    RMSE = mean_squared_error(true_labels, pred_labels,squared=False)
    return mape, MAE, RMSE, r2,
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import LeaveOneOut
from numpy import absolute
from numpy import mean
from numpy import std
from sklearn.model_selection import cross_validate
from sklearn.metrics import mean_squared_error
cv = LeaveOneOut()
Model1.fit(train_features1, train_labels1)
Model2.fit(train_features2,train_labels2)
Model3.fit(train_features3,train_labels3)
from sklearn.model_selection import cross_val_predict
y_pred1 = cross_val_predict(Model1, features, label1, cv=cv)
y_pred2 = cross_val_predict(Model2, features, label2, cv=cv)
y_pred3 = cross_val_predict(Model3, features, label3, cv=cv)
model_accuracy1, model_MAE1, model_RMSE1, model_r21 = evaluate(label1, y_pred1)
model_accuracy2, model_MAE2, model_RMSE2, model_r22 = evaluate(label2, y_pred2)
model_accuracy3, model_MAE3, model_RMSE3, model_r23 = evaluate(label3, y_pred3)
final_output = pd.DataFrame()
final_output.loc[:,1] = [model_accuracy1, model_MAE1, model_RMSE1, model_r21]
final_output.loc[:,2] = [model_accuracy2, model_MAE2, model_RMSE2, model_r22]
final_output.loc[:,3] = [model_accuracy3, model_MAE3, model_RMSE3, model_r23]
final_output=pd.DataFrame(final_output)
final_output=final_output.T
final_output.columns = ['MAPE', 'MAE', 'RMSE', 'R2']
final_output.index = ['UTS', 'TYS', ' Elong']
final_output.to_excel(r'final_output_LOOCV.xlsx', index = True)
#进行力学性能的预测

In [3]:
from sklearn.model_selection import cross_val_predict
#进行力学性能的预测
data=pd.read_excel('data1.xlsx')
original_shuju = data.values
data_shuju = original_shuju[:,:]
data_shuju = MinMaxScaler().fit_transform(data_shuju)
new_data_x_scaled=data_shuju
predicted_y1 = Model1.predict(new_data_x_scaled)
predicted_y2 = Model2.predict(new_data_x_scaled)
predicted_y3 = Model3.predict(new_data_x_scaled)
result_df = pd.DataFrame({'X': new_data_x_scaled.tolist(), '抗拉强度': predicted_y1,
                         '屈服强度': predicted_y2,'延伸率': predicted_y3})#将预测的真实值进行还原
result_df.to_excel('性能预测结果.xlsx', index=False)#保存到表格中